# Day 4

### Tokenizing with code

In [9]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie.")

In [10]:
tokens

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148, 13]

In [11]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id}: {token_text}")

12194: Hi
922:  my
1308:  name
382:  is
6117:  Ed
326:  and
357:  I
1299:  like
9171:  ban
26458: offee
5148:  pie
13: .


In [12]:
encoding.decode([326])

' and'

## And another topic!
### The Illusion of "memory"
Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [13]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API Key was found.")
elif not api_key.startswith('sk-proj-'):
    print("An API Key found, but it not started with \'sk-proj-\'")
elif api_key.strip() != api_key:
    print("An API Key found, but it has some extra spaces or tab at the start or end of the characters.")
else:
    print("An API Key found and looks good so far!!!")

An API Key found and looks good so far!!!


### You should be very comfortable with what the next cell is doing!

I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or
other LLM providers

In [21]:
from openai import OpenAI

openai_client = OpenAI()

### A message to OpenAI is a list of dicts

In [28]:
messages = [
    {'role': 'system', 'content': 'You are an helpful assistant.'},
    {'role': 'user', 'content': "Hi! I'm Ed!"}
]

In [23]:
response = openai_client.chat.completions.create(
    model= "gpt-4.1-mini",
    messages= messages
)

response.choices[0].message.content

'Hello Ed! How can I assist you today?'

### OK let's now ask a follow-up question

In [29]:
messages = [
    {'role': 'system', 'content': "You are a helpful assistant."},
    {'role': 'user', 'content': 'Whats my name?'}
]

In [25]:
response= openai_client.chat.completions.create(
    model= "gpt-4.1-mini",
    messages= messages
)

response.choices[0].message.content

"I don't have access to your name. How can I assist you today?"

### Wait, wha??
We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [30]:
messages = [
    {'role': 'system', 'content': "You are an helpful AI assistant"},
    {'role': 'user', 'content': 'Hello! My name is Ed!'},
    {'role': 'assistant', 'content': "Hello Ed! How can I assist you today?"},
    {'role': 'user', 'content': "What's my name?"}
]

In [31]:
response = openai_client.chat.completions.create(
    model= "gpt-4.1-mini",
    messages= messages
)

response.choices[0].message.content

'Your name is Ed! How can I help you today, Ed?'

### To recap
With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
4. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!